# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_mask = cat['product_name'].str.lower().str.contains('pop', na=False)
pop_df = cat[pop_mask]
print(f"Count: {len(pop_df)}")
pop_df

Count: 3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

SyntaxError: invalid syntax. Perhaps you forgot a comma? (518623731.py, line 2)

**Pop-Tart rows that are a different product:** _Strawberry Pop-Tart SKUs:

PT-12 (pop-tarts strawberry)

SKU#459812 (Strawberry Pop-Tart)

POPTART-STRAW (Pop-Tarts Strawberry)

Other Pop-Tart Rows:the full catalog contains other flavors (e.g., S'mores, Brown Sugar, Cherry) _

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [7]:
# TODO
sku_mapping = {
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12'
}

cat['canonical_sku'] = cat['sku'].replace(sku_mapping)

cat.head()

,sku,product_name,category,canonical_sku,category_clean
0,PT-12,pop-tarts strawberry,Breakfast Foods,PT-12,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast,PT-12,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,PT-12,Breakfast


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [8]:
# TODO
print("--- Before ---")
print(cat['category'].value_counts(dropna=False))

cat['category_clean'] = cat['category'].str.strip().str.title()

print("\n--- After ---")
print(cat['category_clean'].value_counts(dropna=False))

--- Before ---
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

--- After ---
category_clean
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [9]:
# TODO
import string

def get_base_name(name):
    if pd.isna(name): return name
    clean_name = str(name).lower()
    clean_name = clean_name.translate(str.maketrans('', '', string.punctuation))
    words = sorted(clean_name.split())
    return ' '.join(words)

cat['norm_name'] = cat['product_name'].apply(get_base_name)

grouped = cat.groupby('norm_name')['sku'].unique()

fragmented = grouped[grouped.apply(len) > 1]
print(fragmented)

norm_name
poptarts strawberry    [PT-12, POPTART-STRAW]
Name: sku, dtype: object


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [10]:
# TODO
assert cat['canonical_sku'].isnull().sum() == 0, "Error: The mapping introduced NaN values!"
assert len(cat['canonical_sku']) == len(cat['sku']), "Error: The row count changed!"
print("Checks passed! The mapping is safe.")

Checks passed! The mapping is safe.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a. the two product families that typically appear fragmented are Sodas (e.g., 'Coca-Cola 12pk' vs. 'Coke 12-pack') and Paper Goods (e.g., 'Kleenex Tissues' vs. 'Facial Tissues - Kleenex'). I would group them exactly as we did in Q3: by selecting one primary canonical SKU for each and using a dictionary mapping with .replace() to consolidate the messy variants.

b. Consolidation becomes a business decision when items differ by size, count, or brand tier, rather than just spelling. For example, if I see '16oz Peanut Butter' and '24oz Peanut Butter', it is no longer just data cleaning. Merging them might be fine for a Marketing team wanting total brand revenue, but it would be disastrous for an Inventory team tracking physical shelf space. In a real company, I would ask the primary stakeholder requesting the data what level of granularity they require for their specific report.